## Logging — Advanced Setup

### Why move from manual setup to `logging.config.dictConfig()`?

Manual `logger.addHandler(...)` setup works for small scripts. For production services it has problems:

**1. Config mixed with code** — changing a file path or format string means editing Python source.

**2. Partial-state danger** — if the setup block runs more than once (Jupyter re-run, module reload,
test teardown), you accumulate duplicate handlers and get duplicate log lines.

**3. No single source of truth** — the full logging config is scattered across
individual handler and formatter creation calls.

`logging.config.dictConfig()` solves all three:
- **Atomic** — one call, entire config applied at once; no partial state
- **Idempotent** — safe to call again; replaces the previous configuration
- **Separable** — the dict can come from a JSON file, a k8s ConfigMap, or an env var

```python
import json, logging.config
with open("logging_config.json") as f:
    logging.config.dictConfig(json.load(f))
# That's the entire setup call in a production entry point.
```

#### `disable_existing_loggers` — the most important key

| Value | Behaviour |
|---|---|
| `false` **(recommended)** | Loggers already created at import time keep their settings. Safe for library code. |
| `true` | Every logger NOT named in the config is immediately disabled. Useful to silence noisy libraries, but can cause confusing silence in your own code if you forget to list a logger. |


In [1]:
import logging
import logging.config
from pathlib import Path

Path("logs").mkdir(exist_ok=True)

LOGGING = {
    "version": 1,                        # required — only valid value today
    "disable_existing_loggers": False,    # keep loggers created at import time

    "formatters": {
        "simple": {
            "format": "%(levelname)-8s | %(name)s | %(message)s",
        },
        "detailed": {
            "format": "%(asctime)s | %(name)-25s | %(levelname)-8s | %(funcName)s:%(lineno)d | %(message)s",
            "datefmt": "%Y-%m-%d %H:%M:%S",
        },
    },

    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "simple",          # brief for the terminal
            "level": "DEBUG",
            "stream": "ext://sys.stdout",   # ext:// resolves a Python dotted path
        },
        "rotating_file": {
            "class": "logging.handlers.RotatingFileHandler",
            "formatter": "detailed",        # full detail for the file
            "level": "INFO",
            "filename": "logs/app.log",
            "maxBytes": 5_242_880,          # 5 MB per file
            "backupCount": 3,               # keep 3 rotated files → 4 total max
            "encoding": "utf-8",
        },
    },

    "loggers": {
        "myapp": {
            "level": "DEBUG",
            "handlers": ["console", "rotating_file"],
            "propagate": False,             # stop here — don't pass up to root
        },
    },

    "root": {
        "level": "WARNING",               # catch-all for third-party libraries
        "handlers": ["console"],
    },
}

logging.config.dictConfig(LOGGING)

logger = logging.getLogger("myapp")
logger.debug("debug   — console only (rotating_file level is INFO)")
logger.info("info    — console + file")
logger.warning("warning — console + file")
logger.error("error   — console + file")

print()
print(f"logs/app.log exists: {Path('logs/app.log').exists()}")
print()
print("--- logs/app.log contents (written by rotating_file handler) ---")
print(Path("logs/app.log").read_text(encoding="utf-8"))


DEBUG    | myapp | debug   — console only (rotating_file level is INFO)


INFO     | myapp | info    — console + file


WARNING  | myapp | warning — console + file


ERROR    | myapp | error   — console + file



logs/app.log exists: True

--- logs/app.log contents (written by rotating_file handler) ---
2026-07-08 08:57:49 | myapp                     | INFO     | <module>:57 | info    — console + file
2026-07-08 08:57:49 | myapp                     | WARNING  | <module>:58 | warning — console + file
2026-07-08 08:57:49 | myapp                     | ERROR    | <module>:59 | error   — console + file



### JSON config file — separating config from code

The dict passed to `dictConfig()` can come from a JSON file loaded at startup.
This is the standard production pattern:

```shell
myproject/
├── main.py                ← loads config, starts the app
├── logging_config.json    ← the entire logging setup as data
└── myapp/
    ├── orders/views.py    ← logger = logging.getLogger(__name__)
    └── payments/service.py ← logger = logging.getLogger(__name__)
```

```python
# main.py (or wsgi.py / asgi.py) — the ONE place that configures logging
import json, logging.config
from pathlib import Path

with open(Path(__file__).parent / "logging_config.json") as f:
    logging.config.dictConfig(json.load(f))
```

**Benefits:**
- Change levels, file paths, or formats without touching Python source
- Swap config files per environment — dev, staging, prod — via Docker volume or k8s ConfigMap
- Ops can read and adjust the JSON without knowing Python

#### Config file structure at a glance

```json
{
  "version": 1,
  "disable_existing_loggers": false,

  "formatters": {
    "NAME": { "format": "...", "datefmt": "..." }
  },

  "handlers": {
    "NAME": {
      "class": "logging.StreamHandler",
      "formatter": "NAME",
      "level": "DEBUG",
      "stream": "ext://sys.stdout"
    },
    "NAME2": {
      "class": "logging.handlers.RotatingFileHandler",
      "formatter": "NAME",
      "level": "INFO",
      "filename": "logs/app.log",
      "maxBytes": 10485760,
      "backupCount": 3,
      "encoding": "utf-8"
    }
  },

  "loggers": {
    "app": {
      "level": "DEBUG",
      "handlers": ["NAME", "NAME2"],
      "propagate": false
    }
  },

  "root": {
    "level": "WARNING",
    "handlers": ["NAME"]
  }
}
```

#### `ext://` prefix

```json
"stream": "ext://sys.stdout"
```

The `ext://` prefix tells `dictConfig` to resolve the string as a Python dotted path
(`sys.stdout`) rather than treating it as a literal string.
Use it anywhere a config field expects a Python object rather than a plain string value.


In [2]:
import json
import logging
import logging.config
from pathlib import Path

# ── Step 1: write the config JSON ─────────────────────────────────────────────
# In a real project this file already exists in the repo; you just load it.
config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "simple": {
            "format": "%(levelname)-8s | %(name)s | %(message)s"
        },
        "detailed": {
            "format": "%(asctime)s | %(name)-25s | %(levelname)-8s | %(funcName)s:%(lineno)d | %(message)s",
            "datefmt": "%Y-%m-%d %H:%M:%S"
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "simple",
            "level": "DEBUG",
            "stream": "ext://sys.stdout"
        },
        "file_handler": {
            "class": "logging.handlers.RotatingFileHandler",
            "formatter": "detailed",
            "level": "INFO",
            "filename": "logs/app.log",
            "maxBytes": 10485760,
            "backupCount": 3,
            "encoding": "utf-8"
        }
    },
    "loggers": {
        "app": {
            "level": "DEBUG",
            "handlers": ["console", "file_handler"],
            "propagate": False
        }
    },
    "root": {
        "level": "WARNING",
        "handlers": ["console"]
    }
}

config_path = Path("logging_config.json")
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Config written to: {config_path.resolve()}")
print()

# ── Step 2: the production startup call — load JSON, apply dictConfig ──────────
Path("logs").mkdir(exist_ok=True)

with config_path.open() as f:
    logging.config.dictConfig(json.load(f))

# ── Step 3: each module just calls getLogger(__name__) and starts logging ─────
logger = logging.getLogger("app")
child  = logging.getLogger("app.orders")    # child of "app"

logger.debug("debug — console only (file handler level is INFO)")
logger.info("info  — console + file")
child.info("from app.orders — propagates up through app's handlers")
logger.error("error — console + file")

print()
print("--- logs/app.log (file_handler output) ---")
print(Path("logs/app.log").read_text(encoding="utf-8"))


Config written to: C:\Users\Dennie\Mate Academy\Python Core\python_essentials\logging_module\logging_config.json



DEBUG    | app | debug — console only (file handler level is INFO)


INFO     | app | info  — console + file


INFO     | app.orders | from app.orders — propagates up through app's handlers


ERROR    | app | error — console + file



--- logs/app.log (file_handler output) ---
2026-07-08 08:57:49 | myapp                     | INFO     | <module>:57 | info    — console + file
2026-07-08 08:57:49 | myapp                     | WARNING  | <module>:58 | warning — console + file
2026-07-08 08:57:49 | myapp                     | ERROR    | <module>:59 | error   — console + file
2026-07-08 08:57:49 | app                       | INFO     | <module>:66 | info  — console + file
2026-07-08 08:57:49 | app.orders                | INFO     | <module>:67 | from app.orders — propagates up through app's handlers
2026-07-08 08:57:49 | app                       | ERROR    | <module>:68 | error — console + file



### Production module pattern + NullHandler for libraries

#### Every application module — one line

```python
# myapp/orders/views.py
import logging

logger = logging.getLogger(__name__)   # "myapp.orders.views"
# Nothing else. No handler, no formatter, no level.
```

The entry point configures logging ONCE. All modules inherit the config through the hierarchy.

#### NullHandler for library code

If you are writing a **reusable library** (not an application), do NOT configure logging.
Add only a `NullHandler` to your library's package root:

```python
# mylib/__init__.py
import logging
logging.getLogger(__name__).addHandler(logging.NullHandler())
```

**Why this matters:**

Without it, if an application imports your library and hasn't configured any logging,
Python prints a `"No handlers could be found for logger X"` warning to stderr.

With `NullHandler`, your library is logging-aware but hands all control to the application.
The application's `dictConfig()` call can then configure your library's loggers as it sees fit.

This is the convention used by every well-behaved stdlib and popular library:
`httpx`, `sqlalchemy`, `boto3`, `celery` — they all do exactly this.

#### The full picture: who does what

| Role | Who | What they do |
|---|---|---|
| Library | `mylib/__init__.py` | `addHandler(NullHandler())` — nothing else |
| App module | Every `.py` file | `logger = logging.getLogger(__name__)` — nothing else |
| App entry point | `main.py` / `wsgi.py` / `asgi.py` | `dictConfig(...)` — the ONE place |


In [3]:
import logging
import logging.config

LOGGING = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "fmt": {
            "format": "%(levelname)-8s | %(name)-35s | %(message)s",
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "fmt",
            "stream": "ext://sys.stdout",
        }
    },
    "loggers": {
        # Only configure the app root — all children inherit via propagation
        "myapp": {"level": "DEBUG", "handlers": ["console"], "propagate": False},
    },
    "root": {
        "level": "WARNING",    # third-party libs emit at WARNING+
        "handlers": ["console"],
    },
}

logging.config.dictConfig(LOGGING)

# In a real project each of these lines lives in the relevant module file
orders_logger   = logging.getLogger("myapp.orders")
payments_logger = logging.getLogger("myapp.payments")
views_logger    = logging.getLogger("myapp.orders.views")
lib_logger      = logging.getLogger("sqlalchemy.engine")   # simulated library logger

orders_logger.info("order created — routes through myapp handler")
payments_logger.warning("payment retry #2 — same myapp handler")
views_logger.debug("rendering order list — also through myapp")
lib_logger.debug("SA debug — blocked by root level=WARNING, never appears")
lib_logger.warning("SA connection pool warning — passes root level, appears")

print()
print("Effective levels (NOTSET means 'inherit from parent'):")
for name in ["myapp", "myapp.orders", "myapp.orders.views", "myapp.payments"]:
    lg = logging.getLogger(name)
    own  = logging.getLevelName(lg.level)
    eff  = logging.getLevelName(lg.getEffectiveLevel())
    print(f"  {name:<35} own={own:<8} effective={eff}")


INFO     | myapp.orders                        | order created — routes through myapp handler


WARNING  | myapp.payments                      | payment retry #2 — same myapp handler


DEBUG    | myapp.orders.views                  | rendering order list — also through myapp


WARNING  | sqlalchemy.engine                   | SA connection pool warning — passes root level, appears



Effective levels (NOTSET means 'inherit from parent'):
  myapp                               own=DEBUG    effective=DEBUG
  myapp.orders                        own=NOTSET   effective=DEBUG
  myapp.orders.views                  own=NOTSET   effective=DEBUG
  myapp.payments                      own=NOTSET   effective=DEBUG


### `LoggerAdapter` — per-request contextual fields

In a web app or Celery worker, you want a `request_id` or `user_id` attached to
every log line produced during a request — without passing those values as arguments
to every single `logger.info()` call.

`logging.LoggerAdapter` wraps a logger and injects extra fields into every `LogRecord`:

```python
import logging

base = logging.getLogger("myapp.api")

# Inside a request handler (FastAPI dependency / Django middleware):
log = logging.LoggerAdapter(base, extra={"request_id": "abc-123"})

log.info("user authenticated")   # request_id injected automatically
log.error("payment failed")      # no need to pass request_id again
```

The `extra` dict values are merged into `LogRecord` and available to the formatter:

```
"%(asctime)s | %(request_id)s | %(levelname)s | %(message)s"
```

#### `LoggerAdapter` vs `logging.Filter` — when to use each

| | LoggerAdapter | Filter |
|---|---|---|
| Scope | Per-logger instance (per-request context) | Attached to a handler (global) |
| Use case | `request_id`, `user_id`, per-task context | `env`, `hostname`, `app_version` |
| Thread-safe | Yes — each request gets its own adapter instance | Depends on implementation |

#### Alternative: `logging.Filter` for global context

For fields that apply to ALL records (hostname, environment):

```python
class EnvFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        record.env = "production"
        return True   # returning True keeps the record; False drops it

handler.addFilter(EnvFilter())
```


In [4]:
import logging
import logging.config
import uuid

LOGGING = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "request_fmt": {
            "format": "%(asctime)s | req=%(request_id)s | %(levelname)-8s | %(name)s | %(message)s",
            "datefmt": "%H:%M:%S",
        },
        "simple": {
            "format": "%(levelname)-8s | %(name)s | %(message)s",
        },
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "request_fmt",    # expects request_id on every record
            "stream": "ext://sys.stdout",
        },
        "root_console": {
            "class": "logging.StreamHandler",
            "formatter": "simple",         # no request_id — safe for third-party logs
            "stream": "ext://sys.stdout",
        },
    },
    "loggers": {
        "myapp": {
            "level": "DEBUG",
            "handlers": ["console"],
            "propagate": False,            # keep records away from root_console
        },
    },
    "root": {
        "level": "WARNING",
        "handlers": ["root_console"],      # uses simple formatter — no request_id needed
    },
}

logging.config.dictConfig(LOGGING)

base_logger = logging.getLogger("myapp.api")


def handle_request(user_id: int) -> None:
    # Simulates a FastAPI / Django view handling one HTTP request.
    request_id = str(uuid.uuid4())[:8]

    # Every log call inside this function (and anything it calls) goes through the adapter
    log = logging.LoggerAdapter(base_logger, extra={"request_id": request_id})

    log.info(f"request started — user_id={user_id}")
    log.debug("fetching user from db")
    log.info("user authenticated — role=admin")
    log.warning("3 failed logins in the last hour — flagging")
    log.info("request completed — status=200")


print("=== Request 1 ===")
handle_request(user_id=42)

print()
print("=== Request 2 ===")
handle_request(user_id=99)


=== Request 1 ===
08:57:49 | req=2e96e92b | INFO     | myapp.api | request started — user_id=42


08:57:49 | req=2e96e92b | DEBUG    | myapp.api | fetching user from db


08:57:49 | req=2e96e92b | INFO     | myapp.api | user authenticated — role=admin


08:57:49 | req=2e96e92b | WARNING  | myapp.api | 3 failed logins in the last hour — flagging


08:57:49 | req=2e96e92b | INFO     | myapp.api | request completed — status=200



=== Request 2 ===
08:57:49 | req=e340214c | INFO     | myapp.api | request started — user_id=99


08:57:49 | req=e340214c | DEBUG    | myapp.api | fetching user from db


08:57:49 | req=e340214c | INFO     | myapp.api | user authenticated — role=admin


08:57:49 | req=e340214c | WARNING  | myapp.api | 3 failed logins in the last hour — flagging


08:57:49 | req=e340214c | INFO     | myapp.api | request completed — status=200


## Gotchas

### 1. `basicConfig()` is silently ignored after the first handler is added

```python
logging.warning("this call implicitly adds a default StreamHandler to root")
logging.basicConfig(level=logging.DEBUG)   # ← does nothing! root already has a handler
```

Fix: call `basicConfig()` before any logging calls, or use `force=True` (Python 3.8+).

### 2. Re-running a Jupyter cell adds duplicate handlers

```python
logger.addHandler(handler)   # second run → two handlers → every message appears twice
```

Fix: guard with `if not logger.handlers:` OR use `dictConfig()` (it's idempotent).

### 3. `disable_existing_loggers: true` silences library loggers

`sqlalchemy`, `httpx`, `celery`, `boto3` all create loggers at import time.
If `disable_existing_loggers` is `true`, those loggers are disabled immediately
when you call `dictConfig()`, even if the libraries are actively in use.
Always use `false` unless you have a specific reason to silence specific loggers.

### 4. Logger level AND handler level are both filters — both must pass

```
logger.setLevel(INFO)         ← gate 1: DEBUG dropped here, INFO passes
handler.setLevel(WARNING)     ← gate 2: INFO dropped here, WARNING passes
```

DEBUG messages disappearing even when `logger.level = DEBUG`?
Check the handler's level — it may be filtering them out.

---

### Decision tree — which setup to use

```shell
One-off script or quick debug tool
  └──► logging.basicConfig(level=..., format=..., force=True)

Medium project — single process, config lives in Python
  └──► logging.config.dictConfig(LOGGING_DICT)   ← dict in settings.py

Production service (Django / FastAPI / Celery / microservice)
  └──► logging.config.dictConfig(json.load(open("logging_config.json")))
         ├── dev.json   : level=DEBUG, console only
         ├── prod.json  : level=INFO, console + RotatingFileHandler + Sentry/Datadog
         └── mounted via Docker volume, k8s ConfigMap, or env var

Writing a reusable library (not an application)
  └──► logging.getLogger(__name__).addHandler(logging.NullHandler())
         Hand all configuration to the application.
```
